Đọc dữ liệu

In [1]:

# CELL 2: IMPORT CAC THU VIEN

import numpy as np  # Thu vien tinh toan so hoc

import pandas as pd  # Thu vien xu ly du lieu dang bang

from pathlib import Path  # Thu vien xu ly duong dan tep

from sklearn.model_selection import KFold  # Chia du lieu thanh K fold

from sklearn.preprocessing import OrdinalEncoder  # Ma hoa thuoc tinh phan loai

from sklearn.ensemble import RandomForestRegressor  # Mo hinh Random Forest hoi quy

from sklearn.metrics import mean_squared_error  # Tinh MSE

from sklearn.metrics import mean_absolute_error  # Tinh MAE

from sklearn.metrics import r2_score  # Tinh R2

print("Da import cac thu vien thanh cong!")

# CELL 4: DOC DU LIEU GOC

# Duong dan khi notebook chay trong experiments/random_forest
file_path = Path("../../data/raw/Diamonds Prices2022.csv")

# Neu kernel chay tu thu muc goc du an
if not file_path.exists():
    file_path = Path("data/raw/Diamonds Prices2022.csv")

# Kiem tra file co ton tai hay khong
if not file_path.exists():
    raise FileNotFoundError("Khong tim thay file Diamonds Prices2022.csv")

# Doc file CSV, su dung cot dau tien lam chi so dong
diamonds = pd.read_csv(file_path, index_col=0)

# In duong dan dang su dung
print("Duong dan:", file_path.resolve())

# In so dong va so cot ban dau
print("Kich thuoc ban dau:", diamonds.shape)

# Dem so gia tri thieu
print("So gia tri thieu:", diamonds.isnull().sum().sum())

# Dem so dong trung lap
print("So dong trung lap:", diamonds.duplicated().sum())

# Xac dinh cac dong co kich thuoc khong hop le
invalid_size = (
    (diamonds["x"] <= 0) |
    (diamonds["y"] <= 0) |
    (diamonds["z"] <= 0)
)

# In so dong co kich thuoc khong hop le
print("So dong co kich thuoc khong hop le:", invalid_size.sum())

# Hien thi 5 dong dau tien
display(diamonds.head())


Da import cac thu vien thanh cong!
Duong dan: D:\DiamondPrediction\data\raw\Diamonds Prices2022.csv
Kich thuoc ban dau: (53943, 10)
So gia tri thieu: 0
So dong trung lap: 149
So dong co kich thuoc khong hop le: 20


,carat,cut,color,clarity,depth,table,price,x,y,z
1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


Tiền xử lý dữ liệu

In [2]:

# CELL 6: TIEN XU LY DU LIEU

# Tao ban sao cua du lieu goc
df = diamonds.copy()

# Luu so dong ban dau
initial_rows = len(df)

# Loai bo cac dong co gia tri thieu
df = df.dropna()

# Luu so dong sau khi xu ly gia tri thieu
after_missing = len(df)

# Chi giu cac dong co x, y, z lon hon 0
df = df[
    (df["x"] > 0) &
    (df["y"] > 0) &
    (df["z"] > 0)
]

# Luu so dong sau khi loc kich thuoc
after_dimensions = len(df)

# Xoa cac dong trung lap
df = df.drop_duplicates()

# Dat lai chi so dong tu 0
df = df.reset_index(drop=True)

# In so dong bi loai do thieu du lieu
print("Dong thieu bi loai:", initial_rows - after_missing)

# In so dong co kich thuoc khong hop le bi loai
print("Dong kich thuoc khong hop le bi loai:",
      after_missing - after_dimensions)

# In so dong trung lap bi loai
print("Dong trung lap bi loai:",
      after_dimensions - len(df))

# In so dong con lai
print("So dong sau tien xu ly:", len(df))

# Kiem tra so gia tri thieu con lai
print("Gia tri thieu con lai:", df.isnull().sum().sum())

# Kiem tra so dong trung lap con lai
print("Dong trung lap con lai:", df.duplicated().sum())

# Hien thi 5 dong dau tien
display(df.head())


Dong thieu bi loai: 0
Dong kich thuoc khong hop le bi loai: 20
Dong trung lap bi loai: 148
So dong sau tien xu ly: 53775
Gia tri thieu con lai: 0
Dong trung lap con lai: 0


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


Khởi tạo K-Fold

In [3]:

# CELL 8: CHUAN BI DU LIEU VA K-FOLD

# Lay tat ca cac cot tru price lam dac trung
X = df.drop(columns=["price"])

# Lay cot price lam bien muc tieu
y = df["price"]

# Khoi tao bo chia 5-Fold
kf = KFold(
    n_splits=5,  # Chia thanh 5 fold
    shuffle=True,  # Tron ngau nhien truoc khi chia
    random_state=42  # Co dinh cach chia
)

# In tong so mau sau tien xu ly
print("Tong so mau:", len(X))

# In so fold
print("So fold:", kf.get_n_splits())

# In so luong dac trung dau vao
print("So dac trung:", X.shape[1])


Tong so mau: 53775
So fold: 5
So dac trung: 9


Huấn luyện 5-Fold

In [4]:

# CELL 10: HUAN LUYEN RANDOM FOREST 5-FOLD

# Thu tu chat luong mau sac tu thap den cao
color_order = ["J", "I", "H", "G", "F", "E", "D"]

# Thu tu do tinh khiet tu thap den cao
clarity_order = [
    "I1", "SI2", "SI1", "VS2",
    "VS1", "VVS2", "VVS1", "IF"
]

# Thu tu chat luong giac cat tu thap den cao
cut_order = [
    "Fair", "Good", "Very Good",
    "Premium", "Ideal"
]

# Danh sach cac cot can ma hoa
categorical_cols = ["color", "clarity", "cut"]

# Tao danh sach rong de luu ket qua
results = []

# Lap qua tung fold cua KFold
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):

    # Lay du lieu dau vao cua Train
    X_train = X.iloc[train_idx].copy()

    # Lay du lieu dau vao cua Test
    X_test = X.iloc[test_idx].copy()

    # Lay gia kim cuong thuc te cua Train
    y_train = y.iloc[train_idx]

    # Lay gia kim cuong thuc te cua Test
    y_test = y.iloc[test_idx]

    # Khoi tao bo ma hoa moi cho tung fold
    encoder = OrdinalEncoder(
        categories=[
            color_order,
            clarity_order,
            cut_order
        ]
    )

    # Hoc bo ma hoa tren Train va chuyen thanh so
    X_train[categorical_cols] = encoder.fit_transform(
        X_train[categorical_cols]
    )

    # Ma hoa Test bang bo ma hoa da hoc tren Train
    X_test[categorical_cols] = encoder.transform(
        X_test[categorical_cols]
    )

    # Khoi tao Random Forest voi tham so hoc mac dinh
    model = RandomForestRegressor(
        random_state=42,  # Co dinh tinh ngau nhien
        n_jobs=-1  # Su dung cac loi CPU kha dung
    )

    # Huan luyen mo hinh tren Train cua fold
    model.fit(X_train, y_train)

    # Du doan gia kim cuong tren Test cua fold
    y_pred = model.predict(X_test)

    # Tinh sai so binh phuong trung binh
    mse = mean_squared_error(y_test, y_pred)

    # Tinh can bac hai cua MSE
    rmse = np.sqrt(mse)

    # Tinh sai so tuyet doi trung binh
    mae = mean_absolute_error(y_test, y_pred)

    # Tinh he so xac dinh R2
    r2 = r2_score(y_test, y_pred)

    # Luu ket qua cua fold hien tai
    results.append({
        "Fold": fold,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    })

    # In ket qua RMSE cua fold hien tai
    print(f"Fold {fold}: RMSE = {rmse:.4f}")

# Chuyen danh sach ket qua thanh DataFrame
results_df = pd.DataFrame(results)

# Hien thi ket qua cua 5 fold
display(results_df.round(4))


Fold 1: RMSE = 526.8399
Fold 2: RMSE = 533.6411
Fold 3: RMSE = 539.0144
Fold 4: RMSE = 537.4932
Fold 5: RMSE = 569.1761


,Fold,MSE,RMSE,MAE,R2
0,1,277560.2606,526.8399,267.2045,0.9824
1,2,284772.7974,533.6411,264.9074,0.9817
2,3,290536.5176,539.0144,264.3154,0.9817
3,4,288898.9018,537.4932,267.8643,0.9822
4,5,323961.3852,569.1761,271.1122,0.9797


Tổng hợp kết quả

In [5]:

# CELL 12: TONG HOP KET QUA 5-FOLD

# Danh sach cac chi so can tong hop
metrics = ["MSE", "RMSE", "MAE", "R2"]

# Tinh trung binh cua tung chi so
mean_scores = results_df[metrics].mean()

# Tinh do lech chuan mau cua tung chi so
std_scores = results_df[metrics].std()

# Tao bang tong hop ket qua
summary = pd.DataFrame({
    "Chi so": metrics,  # Ten cac chi so
    "Trung binh": mean_scores.values,  # Gia tri trung binh
    "Do lech chuan": std_scores.values  # Do lech chuan
})

# Hien thi bang tong hop
display(summary.round(4))

# In ten thuc nghiem
print("MUC 2 - RANDOM FOREST 5-FOLD")

# In duong phan cach
print("-" * 45)

# In MSE trung binh
print(f"MSE trung binh : {mean_scores['MSE']:.4f}")

# In RMSE trung binh
print(f"RMSE trung binh: {mean_scores['RMSE']:.4f}")

# In do lech chuan RMSE
print(f"RMSE do lech chuan: {std_scores['RMSE']:.4f}")

# In MAE trung binh
print(f"MAE trung binh : {mean_scores['MAE']:.4f}")

# In R2 trung binh
print(f"R2 trung binh  : {mean_scores['R2']:.4f}")


,Chi so,Trung binh,Do lech chuan
0,MSE,293145.9725,17940.6034
1,RMSE,541.2329,16.3135
2,MAE,267.0808,2.7043
3,R2,0.9815,0.0011


MUC 2 - RANDOM FOREST 5-FOLD
---------------------------------------------
MSE trung binh : 293145.9725
RMSE trung binh: 541.2329
RMSE do lech chuan: 16.3135
MAE trung binh : 267.0808
R2 trung binh  : 0.9815


So sánh mức 1 và mức 2

In [6]:

# CELL 14: SO SANH MUC 1 VA MUC 2 K-FOLD

# Ket qua K-Fold muc 1 da chay truoc do
level1_rmse = 543.4347  # RMSE trung binh muc 1

level1_mae = 267.1485  # MAE trung binh muc 1

level1_r2 = 0.9814  # R2 trung binh muc 1

# Tao bang so sanh cac chi so
comparison = pd.DataFrame({
    "Chi so": ["RMSE", "MAE", "R2"],
    "Muc 1": [level1_rmse, level1_mae, level1_r2],
    "Muc 2": [
        mean_scores["RMSE"],  # RMSE trung binh muc 2
        mean_scores["MAE"],  # MAE trung binh muc 2
        mean_scores["R2"]  # R2 trung binh muc 2
    ]
})

# Tinh chenh lech Muc 2 tru Muc 1
comparison["Chenh lech"] = (
    comparison["Muc 2"] - comparison["Muc 1"]
)

# Hien thi bang so sanh
display(comparison.round(4))

# Tinh phan tram thay doi RMSE
rmse_change = (
    (mean_scores["RMSE"] - level1_rmse)
    / level1_rmse * 100
)

# In phan tram thay doi
print(f"Thay doi RMSE: {rmse_change:+.2f}%")

# Kiem tra RMSE tang hay giam
if mean_scores["RMSE"] < level1_rmse:
    print("RMSE trung binh Muc 2 thap hon Muc 1.")

elif mean_scores["RMSE"] > level1_rmse:
    print("RMSE trung binh Muc 2 cao hon Muc 1.")

else:
    print("RMSE trung binh hai muc bang nhau.")


,Chi so,Muc 1,Muc 2,Chenh lech
0,RMSE,543.4347,541.2329,-2.2018
1,MAE,267.1485,267.0808,-0.0677
2,R2,0.9814,0.9815,0.0001


Thay doi RMSE: -0.41%
RMSE trung binh Muc 2 thap hon Muc 1.
